# Part 2 v4 — XGBoost Tweedie (year_built as Numeric)

Same as v3 no CSC but with year_built kept as numeric (not categorical).

**Key changes vs v2:**
- Adds 5 previously-excluded features: discount_applied, written_premium, marketing_consent, document_preference, vehicle_colour
- Adds engine_size_cc, branch_id, customer_since_year (from email guidance)
- customer_service_contacts EXCLUDED (confirmed leakage)
- year_built kept NUMERIC (160 levels too many for categorical with ~1K claims)
- Other categorical fixes kept: vehicle_value_group, engine_size_cc, branch_id, flood_zone, customer_since_year
- Same data cleaning: occupation cleanup, mode imputation, campaign_code dropped
- Same model params as v2: 50 trees, depth=2, min_child_weight=50

In [0]:
# Step 1: Load processed training data and define curated feature sets
import pandas as pd
import numpy as np
import os, pickle

%pip install xgboost openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data_emerging/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data (from Notebook 01 — same as v1)
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape}, target mean={train[line]["ultimate_claims"].mean():.4f}')

# === v3 DATA CLEANING (from email guidance) ===
# 1. Occupation cleanup: lowercase, strip whitespace, remove "self-employed " prefix
# 2. Categorical fix: convert numeric-but-categorical columns to string for GLM
# 3. Mode imputation for ~12% missing values
# 4. Drop campaign_code (~80% missing)

CATEGORICAL_FIX = ['vehicle_value_group', 'engine_size_cc', 'branch_id',
                    'flood_zone', 'customer_since_year']
MODE_IMPUTE = ['overnight_parking', 'occupation', 'marital_status', 'soil_type',
               'roof_type', 'energy_rating', 'smoke_alarms', 'security_level']

def clean_occupation(s):
    """Clean occupation: lowercase, strip, remove 'self-employed ' prefix."""
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    if s.startswith('self-employed '):
        s = s.replace('self-employed ', '')
    return s

def clean_data(df):
    """Apply v3 data cleaning."""
    df = df.copy()
    # Drop campaign_code (~80% missing)
    if 'campaign_code' in df.columns:
        df.drop('campaign_code', axis=1, inplace=True)
    # Occupation cleanup
    if 'occupation' in df.columns:
        df['occupation'] = df['occupation'].apply(clean_occupation)
    # Convert numeric-but-categorical to string (for GLM)
    for col in CATEGORICAL_FIX:
        if col in df.columns:
            df[col] = df[col].astype(str)
    # Mode imputation for ~12% missing
    for col in MODE_IMPUTE:
        if col in df.columns and df[col].isna().any():
            mode_val = df[col].mode()
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
    return df

# Apply cleaning to training data
for line in ['motor', 'buildings', 'contents']:
    train[line] = clean_data(train[line])
    print(f'{line} v3 cleaned: {train[line].shape}')

# Load test files and enrichment
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Trend factors from v1 (reused)
with open(os.path.join(PROC_DIR, 'xgboost_v1_results.pkl'), 'rb') as f:
    v1_results = pickle.load(f)
TREND_FACTORS = v1_results['trend_factors']
print(f'\nTrend factors: {TREND_FACTORS}')

# Curated feature sets — v3 all features trial
# v2 base (13) + 6 previously-excluded features + 3 from email categorical guidance
FEATURE_SETS = {
    'motor': [
        'exposure', 'Year', 'driver_age', 'ncb_years', 'vehicle_value_group',
        'vehicle_age', 'cover_type', 'fuel_type', 'channel',
        'missed_payments_12m', 'prior_claims_5y', 'urban_density', 'renewal_count',
        # v3 new features (previously excluded)
        'discount_applied', 'written_premium',
        'marketing_consent', 'document_preference', 'vehicle_colour',
        # v3 from email categorical guidance
        'engine_size_cc', 'branch_id', 'customer_since_year',
    ],
    'buildings': [
        'exposure', 'Year', 'sum_insured', 'construction_type', 'voluntary_excess',
        'missed_payments_12m', 'prior_claims_5y', 'flood_zone', 'elevation_m',
        'year_built', 'occupancy', 'channel', 'new_business',
        # v3 new features (previously excluded)
        'discount_applied', 'written_premium',
        'marketing_consent', 'document_preference',
        # v3 from email categorical guidance
        'branch_id', 'customer_since_year',
    ],
    'contents': [
        'exposure', 'Year', 'sum_insured', 'security_level', 'voluntary_excess',
        'missed_payments_12m', 'prior_claims_5y', 'flood_zone', 'elevation_m',
        'urban_density', 'channel', 'accidental_damage_cover', 'tenure',
        # v3 new features (previously excluded)
        'discount_applied', 'written_premium',
        'marketing_consent', 'document_preference',
        # v3 from email categorical guidance
        'branch_id', 'customer_since_year',
    ],
}

for line in ['motor', 'buildings', 'contents']:
    feats = FEATURE_SETS[line]
    available = [f for f in feats if f in train[line].columns]
    missing = [f for f in feats if f not in train[line].columns]
    FEATURE_SETS[line] = available
    status = f'{len(available)}/{len(feats)} features available'
    if missing:
        status += f' (MISSING: {missing})'
    print(f'{line}: {status}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment)
AGE_BAND_MAP = {'17-24': 20.5, '25-34': 29.5, '35-49': 42.0, '50-64': 57.0, '65+': 70.0}

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    for col in df.columns:
        if col.endswith('_age_band') or col == 'age_band':
            new_col = col.replace('_age_band', '_age')
            df[new_col] = df[col].map(AGE_BAND_MAP)
    if 'compulsory_excess' in df.columns and df['compulsory_excess'].nunique() <= 1:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Zero-pad customer_id
cust_lens = customers['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    max_len = cust_lens.index.max()
    customers['customer_id'] = customers['customer_id'].str.zfill(max_len)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    if 'customer_id' in df.columns:
        lens = df['customer_id'].str.len().value_counts()
        if lens.nunique() > 1:
            max_len = lens.index.max()
            df['customer_id'] = df['customer_id'].str.zfill(max_len)

# Join enrichment
customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {df.shape}')

# Apply v3 cleaning to test data (after enrichment)
for line in ['motor', 'buildings', 'contents']:
    test[line] = clean_data(test[line])
    print(f'test_{line} v3 cleaned: {test[line].shape}')

In [0]:
# Step 3: Fit XGBoost v3 — all features trial (same params as v2)
import xgboost as xgb
from sklearn.model_selection import KFold

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def prepare_features(train_df, test_df, feature_list):
    available = [f for f in feature_list if f in train_df.columns]
    X_train = train_df[available].copy()
    X_test = test_df[available].copy() if test_df is not None else None
    for col in available:
        if X_train[col].dtype == 'object' or X_train[col].dtype == 'bool':
            cats = pd.Categorical(X_train[col])
            X_train[col] = cats.codes.astype('float64')
            if X_test is not None:
                X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
            X_train[col] = X_train[col].replace(-1, 0)
            if X_test is not None:
                X_test[col] = X_test[col].replace(-1, 0)
    return X_train, X_test, available

# v2 parameters: drastically simplified
XGB_PARAMS = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'max_depth': 2,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 50,
    'reg_alpha': 2.0,
    'reg_lambda': 5.0,
    'seed': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

N_FOLDS = 5
NUM_BOOST_ROUND = 50

cv_scores = {}
oof_predictions = {}
test_predictions = {}
feature_importances = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} XGBoost v4 ===')
    print(f'{"="*60}')

    train_df = train[line]
    test_df = test[line]
    X_all, X_test, feat_cols = prepare_features(train_df, test_df, FEATURE_SETS[line])
    y_all = train_df['ultimate_claims'].values
    exposure_all = train_df['exposure'].values
    exp_test = test_df['exposure'].values
    n_claims = (y_all > 0).sum()
    n_leaf = NUM_BOOST_ROUND * (2**XGB_PARAMS['max_depth'])

    print(f'  Features: {len(feat_cols)} | Trees: {NUM_BOOST_ROUND} | Depth: {XGB_PARAMS["max_depth"]} | MCW: {XGB_PARAMS["min_child_weight"]}')
    print(f'  Effective leaf nodes: ~{n_leaf} vs {n_claims:,} claims (ratio {n_leaf/n_claims:.2f}:1)')

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    fold_scores = []
    oof_pred = np.zeros(len(train_df))
    fold_importances = np.zeros(len(feat_cols))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_all)):
        X_tr, X_val = X_all.iloc[tr_idx], X_all.iloc[val_idx]
        y_tr, y_val = y_all[tr_idx], y_all[val_idx]
        exp_tr, exp_val = exposure_all[tr_idx], exposure_all[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr, base_margin=np.log(exp_tr + 1e-10), weight=exp_tr)
        dval = xgb.DMatrix(X_val, label=y_val, base_margin=np.log(exp_val + 1e-10), weight=exp_val)

        model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=NUM_BOOST_ROUND,
                          evals=[(dval, 'val')], early_stopping_rounds=20, verbose_eval=False)

        val_pred = np.maximum(model.predict(dval), 0)
        d2 = tweedie_d2(y_val, val_pred, p=1.5)
        fold_scores.append(d2)
        oof_pred[val_idx] = val_pred

        scores = model.get_score(importance_type='gain')
        for i, f in enumerate(feat_cols):
            fold_importances[i] += scores.get(f, 0)
        print(f'  Fold {fold+1}: D2 = {d2:.4f}, pred mean = {val_pred.mean():.1f} (actual {y_val.mean():.1f})')

    cv_scores[line] = np.mean(fold_scores)
    oof_predictions[line] = oof_pred
    print(f'  CV mean D2: {cv_scores[line]:.4f} (+/- {np.std(fold_scores):.4f})')
    print(f'  OOF pred mean: {oof_pred.mean():.4f} (actual mean: {y_all.mean():.4f})')

    imp = pd.DataFrame({'feature': feat_cols, 'importance': fold_importances / N_FOLDS})
    imp = imp.sort_values('importance', ascending=False)
    feature_importances[line] = imp
    print(f'  Top 5: {imp.head(5)["feature"].tolist()}')

    # Final model for test predictions
    dtrain_full = xgb.DMatrix(X_all, label=y_all, base_margin=np.log(exposure_all + 1e-10), weight=exposure_all)
    dtest = xgb.DMatrix(X_test, base_margin=np.log(exp_test + 1e-10))
    final_model = xgb.train(XGB_PARAMS, dtrain_full, num_boost_round=NUM_BOOST_ROUND, verbose_eval=False)
    test_pred = np.maximum(final_model.predict(dtest), 0.01)
    test_predictions[line] = test_pred
    print(f'  Test predictions: mean={test_pred.mean():.4f}')

print(f'\n{"="*60}')
print(f'=== v4 vs v1 Summary ===')
for line in ['motor', 'buildings', 'contents']:
    v1_d2 = v1_results['cv_scores'][line]
    print(f'  {line}: v4={cv_scores[line]:.4f} (v1={v1_d2:.4f}, diff={cv_scores[line]-v1_d2:+.4f})')
print(f'  Average: v4={np.mean(list(cv_scores.values())):.4f} (v1={np.mean(list(v1_results["cv_scores"].values())):.4f})')

In [0]:
# Step 4: Save v3 results for Notebook 03
results = {
    'cv_scores': cv_scores,
    'oof_predictions': oof_predictions,
    'test_predictions': test_predictions,
    'feature_importances': feature_importances,
    'trend_factors': TREND_FACTORS,
    'feature_sets': FEATURE_SETS,
    'params': XGB_PARAMS,
    'n_boost_round': NUM_BOOST_ROUND,
}

with open(os.path.join(PROC_DIR, 'xgboost_v4_results.pkl'), 'wb') as f:
    pickle.dump(results, f)

EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_emerging_v4/'
os.makedirs(EXCEL_DIR, exist_ok=True)

# Load v2 results for comparison
try:
    with open(os.path.join(PROC_DIR, 'xgboost_v2_results.pkl'), 'rb') as f:
        v2_results = pickle.load(f)
    v2_scores = v2_results['cv_scores']
except:
    v2_scores = None

with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'xgboost_v4_results.xlsx'), engine='openpyxl') as writer:
    rows = []
    for line in ['motor', 'buildings', 'contents']:
        row = {'line': line, 'v3_d2': cv_scores[line],
               'n_features': len(FEATURE_SETS[line]), 'n_trees': NUM_BOOST_ROUND,
               'max_depth': XGB_PARAMS['max_depth'], 'min_child_weight': XGB_PARAMS['min_child_weight']}
        if v2_scores:
            row['v2_d2'] = v2_scores[line]
            row['diff_v3_v2'] = cv_scores[line] - v2_scores[line]
        row['v1_d2'] = v1_results['cv_scores'][line]
        row['diff_v3_v1'] = cv_scores[line] - v1_results['cv_scores'][line]
        rows.append(row)
    comparison = pd.DataFrame(rows)
    comparison.to_excel(writer, sheet_name='v3 comparison', index=False)
    for line in ['motor', 'buildings', 'contents']:
        feature_importances[line].to_excel(writer, sheet_name=f'{line} importance', index=False)

print(f'Saved: {PROC_DIR}xgboost_v4_results.pkl')
print(f'Saved: {EXCEL_DIR}xgboost_v4_results.xlsx')
print(f'\n{comparison.to_string(index=False)}')